#INIT

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


# Read Orders

In [0]:
orders_path = "abfss://customer360@stcustomers360dev01.dfs.core.windows.net/bronze/sql/dbo.orders.csv"

df_orders = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(orders_path)
)

display(df_orders)

# Check the data

In [0]:
print("Bronze order count:", df_orders.count())

df_orders.printSchema()

# Clean Orders

In [0]:
df_orders_clean = (
    df_orders
    .dropDuplicates()
    .dropDuplicates(["order_id"])
    .filter(col("order_id").isNotNull())
    .filter(col("customer_id").isNotNull())
    .filter(col("product_id").isNotNull())
    .filter(col("store_id").isNotNull())
    
    .withColumn("order_id", col("order_id").cast("int"))
    .withColumn("customer_id", col("customer_id").cast("int"))
    .withColumn("product_id", upper(trim(col("product_id"))))
    .withColumn("store_id", upper(trim(col("store_id"))))
    .withColumn("order_date", to_timestamp(col("order_date")))
    .withColumn("quantity", col("quantity").cast("int"))
    .withColumn("unit_price", col("unit_price").cast("decimal(10,2)"))
    .withColumn("total_amount", col("total_amount").cast("decimal(12,2)"))
    .withColumn("payment_method", trim(col("payment_method")))
    .withColumn("order_status", trim(col("order_status")))
)

# Validate quantity and amount

In [0]:
print("Invalid quantity:")
display(
    df_orders_clean.filter(col("quantity") <= 0)
)

print("Amount mismatches:")
display(
    df_orders_clean.filter(
        col("total_amount") !=
        (col("quantity") * col("unit_price")).cast("decimal(12,2)")
    )
)

# Validate customer IDs, product IDs, and store IDs

## read the customers, products and stores clean data

In [0]:
customers_silver_path = "abfss://customer360@stcustomers360dev01.dfs.core.windows.net/silver/customers/"

df_customers = (
    spark.read
    .format("delta")
    .load(customers_silver_path)
)

products_silver_path = "abfss://customer360@stcustomers360dev01.dfs.core.windows.net/silver/products/"

df_products = (
    spark.read
    .format("delta")
    .load(products_silver_path)
)

stores_silver_path = "abfss://customer360@stcustomers360dev01.dfs.core.windows.net/silver/stores/"

df_stores = (
    spark.read
    .format("delta")
    .load(stores_silver_path)
)

display(df_stores)

In [0]:
# Customer IDs
invalid_customers = df_orders_clean.join(
    df_customers,
    on="customer_id",
    how="left_anti"
)

display(invalid_customers)

print("Invalid customer references:", invalid_customers.count())

# Products Ids
invalid_products = df_orders_clean.join(
    df_products,
    on="product_id",
    how="left_anti"
)

display(invalid_products)

print("Invalid product references:", invalid_products.count())

# Store IDs

invalid_stores = df_orders_clean.join(
    df_stores,
    on="store_id",
    how="left_anti"
)

display(invalid_stores)

print("Invalid store references:", invalid_stores.count())

# Write Orders to Silver as Delta

In [0]:
orders_silver_path = "abfss://customer360@stcustomers360dev01.dfs.core.windows.net/silver/orders/"

(
    df_orders_clean.write
    .format("delta")
    .mode("overwrite")
    .save(orders_silver_path)
)

# Verify

In [0]:
df_orders_silver = (
    spark.read
    .format("delta")
    .load(orders_silver_path)
)

display(df_orders_silver)

print("Silver order count:", df_orders_silver.count())